# 04 流式输出

通过 `stream=True` 获取逐块返回的实时输出。

In [5]:
import sys
sys.path.insert(0, '..')

from src.client import get_client
from src.streaming import print_stream

client = get_client()

## 基础流式调用

In [6]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "请用 200 字左右介绍深度学习。"}
    ],
    stream=True,
)

print_stream(response)

深度学习是人工智能的一个核心分支，它通过模拟人脑神经网络的多层结构，让计算机自动从海量数据中“学习”复杂规律。其关键在于“深层”架构——由输入层、多个隐藏层和输出层组成，每一层都对数据进行逐级抽象。例如，识别一张猫的图片：底层识别边缘，中层识别纹理或形状，高层则组合出猫的整体特征。训练时，模型借助反向传播算法和海量数据不断调整内部参数，使预测误差最小化。凭借强大的特征提取能力，深度学习已广泛赋能语音识别、图像分类、自然语言处理及自动驾驶等领域，推动技术迈向更高层次的智能应用。


## 流式 + 累积内容

In [7]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "用一句话解释什么是 API。"}
    ],
    stream=True,
)

full_text = ""
for chunk in response:
    delta = chunk.choices[0].delta
    if delta.content:
        full_text += delta.content
        print(delta.content, end="", flush=True)

print()
print(f"\n共 {len(full_text)} 个字符")

API 就像餐厅里的服务员——你告诉服务员你要什么（请求），他去厨房（后端系统）帮你处理好，再把菜（数据或结果）端给你，而你完全不需要操心厨房里是怎么做的。

共 78 个字符


## 对流式响应做多轮对话

In [8]:
messages = [
    {"role": "system", "content": "你是一个简洁的助手。"},
    {"role": "user", "content": "什么是图灵测试？"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    stream=True,
)

assistant_text = ""
print("Assistant: ", end="", flush=True)
for chunk in response:
    delta = chunk.choices[0].delta
    if delta.content:
        assistant_text += delta.content
        print(delta.content, end="", flush=True)

messages.append({"role": "assistant", "content": assistant_text})
messages.append({"role": "user", "content": "能用一句话总结吗？"})

print()
print()
print("--- 第二轮 ---")
print_stream(client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    stream=True,
))

Assistant: 图灵测试是由艾伦·图灵在1950年提出的一种测试机器智能的方法。测试中，一名人类评判者通过文本对话与两个参与者（一个是人，一个是机器）交流，如果评判者无法可靠区分哪个是机器，就说明机器具有人类的智能水平。简单说，就是判断机器能否“像人一样思考”。

--- 第二轮 ---
图灵测试是判断机器能否在对话中让人类无法分辨其是否为人的智能测试。
